# Subnational Data Merge Pipeline v2 (KEN + SOM)

**Changes from v1:**
- Target countries: KEN + SOM only (ETH removed — no price data in source CSV)
- Price columns: `c_maize_fao`, `c_food_price_index`, `c_sorghum` (v1 used `c_maize` which was empty for KEN)
- Spatial join: per-country ISO3 filtering to prevent cross-boundary misassignment
- Quality check section added

**Files:**
- `v1_original`: `subnational_merged_v1_original.parquet` / `subnational_merge_notebook_v1_original.ipynb`
- `v2 (this)`: `subnational_merged_v2_KEN_SOM.parquet` / `subnational_merge_notebook.ipynb`


## 0. Setup

In [ ]:
import os
import logging
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
from rasterstats import zonal_stats
import glob
import difflib
from shapely.geometry import Point
import itertools

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

DATA_DIR = "data"
iso3_list = ['KEN', 'SOM']
boundary_dir = os.path.join(DATA_DIR, 'geoboundaries')

print("Environment ready.")
print(f"Target countries: {iso3_list}")

## 1. Helper Functions

In [ ]:
def spatial_join_points(df, gdf, lon_col, lat_col):
    """Spatially join point data to Admin2 polygons."""
    points = gpd.GeoDataFrame(
        df, geometry=gpd.points_from_xy(df[lon_col], df[lat_col]), crs="EPSG:4326"
    )
    if gdf.crs != points.crs:
        gdf = gdf.to_crs(points.crs)
    joined = gpd.sjoin(points, gdf, how="left", predicate="within")
    joined.rename(columns={'shapeName': 'admin2_canonical'}, inplace=True)
    return joined

def fuzzy_match_names(series, choices, threshold=0.8):
    """Map names to canonical choices using fuzzy matching (difflib)."""
    mapping = {}
    unique_names = series.dropna().unique()
    for name in unique_names:
        matches = difflib.get_close_matches(str(name), choices, n=1, cutoff=threshold)
        if matches:
            mapping[name] = matches[0]
    return mapping

def process_worldpop_population(country_gdf, raster_path, iso_code):
    """Compute population per Admin2 using zonal statistics on WorldPop raster."""
    logger.info(f"Processing WorldPop for {iso_code} with {len(country_gdf)} regions...")
    country_gdf = country_gdf[country_gdf['shapeISO'] == iso_code].copy()
    if country_gdf.empty:
        return pd.DataFrame()
    stats = zonal_stats(country_gdf, raster_path, stats="sum", all_touched=True)
    country_gdf['population'] = [s['sum'] for s in stats]
    return country_gdf[['shapeName', 'population', 'shapeISO']].copy()

def load_canonical_boundaries(iso3_list, boundary_dir):
    """Load Admin2 boundaries for specified countries."""
    gdfs = []
    for iso in iso3_list:
        path = os.path.join(boundary_dir, f'gb_{iso}_ADM2.geojson')
        if os.path.exists(path):
            gdf = gpd.read_file(path)
            if 'shapeISO' not in gdf.columns:
                gdf['shapeISO'] = iso
            gdf = gdf[['shapeName', 'shapeISO', 'geometry']]
            gdfs.append(gdf)
            logger.info(f"Loaded {iso}: {len(gdf)} admin2 regions")
        else:
            logger.error(f"Not found: {path}")
    return pd.concat(gdfs, ignore_index=True)

def create_master_skeleton(years, months, admin_gdf):
    """Create master DataFrame with all Year x Month x Admin2 combinations."""
    records = []
    regions = admin_gdf[['shapeName', 'shapeISO']].drop_duplicates()
    for year in years:
        for month in months:
            step = regions.copy()
            step['year'] = year
            step['month'] = month
            records.append(step)
    master = pd.concat(records, ignore_index=True)
    master.rename(columns={'shapeName': 'admin2', 'shapeISO': 'country_iso'}, inplace=True)
    return master

print("Helper functions defined.")

## 2. Load Raw Data

In [ ]:
# Price data (WorldBank imputed)
price_df = pd.read_csv(
    os.path.join(DATA_DIR, 'worldbank_imputed_price_data/WLD_RTFP_mkt_2026-01-13.csv'),
    low_memory=False
)
price_df = price_df[price_df['ISO3'].isin(iso3_list)]
print(f"Price data: {len(price_df)} rows, {price_df['ISO3'].value_counts().to_dict()}")

# Crop mask (aggregated to admin2)
crop_df = pd.read_parquet(os.path.join(DATA_DIR, 'crop_mask/admin_mapped/admin_agg.parquet'))
print(f"Crop data: {len(crop_df)} rows")

# ACLED conflict data
acled_df = pd.read_excel(os.path.join(DATA_DIR, 'raw/acled/Africa_aggregated_data_up_to-2026-01-03.xlsx'))
acled_df = acled_df[acled_df['COUNTRY'].isin(['Kenya', 'Somalia'])]
print(f"ACLED data: {len(acled_df)} rows")

## 3. Load Canonical Boundaries

In [ ]:
admin_gdf = load_canonical_boundaries(iso3_list, boundary_dir)
canonical_names = admin_gdf['shapeName'].unique().tolist()
print(f"Total admin2 regions: {len(canonical_names)}")
print(f"  KEN: {len(admin_gdf[admin_gdf['shapeISO']=='KEN'])}")
print(f"  SOM: {len(admin_gdf[admin_gdf['shapeISO']=='SOM'])}")

## 4. Process Price Data (Spatial Join — per country)

**Key fix (v2):** Spatial join is done per-country to prevent cross-boundary misassignment.  
In v1, SOM markets Doolow (4.16, 42.08) and Wadajir (5.00, 45.00) fell inside ETH polygons.

**Price columns:** `c_maize_fao` (available for both KEN & SOM), `c_food_price_index`, `c_sorghum`  
(v1 used `c_maize` which was empty for KEN)


In [ ]:
# Per-country spatial join to prevent cross-boundary contamination
price_cols = ['c_maize_fao', 'c_food_price_index', 'c_sorghum']
price_joined_parts = []

for iso in iso3_list:
    iso_prices = price_df[price_df['ISO3'] == iso].dropna(subset=['lat', 'lon'])
    iso_bounds = admin_gdf[admin_gdf['shapeISO'] == iso]
    
    if iso_prices.empty:
        logger.warning(f"No price data with coordinates for {iso}")
        continue
    
    joined = spatial_join_points(iso_prices, iso_bounds, 'lon', 'lat')
    matched = joined['admin2_canonical'].notna().sum()
    unmatched = joined['admin2_canonical'].isna().sum()
    print(f"{iso}: {matched}/{len(joined)} matched ({matched/len(joined)*100:.1f}%), {unmatched} unmatched")
    
    # Show unmatched markets
    if unmatched > 0:
        unmatch_mkts = joined[joined['admin2_canonical'].isna()][['mkt_name','lat','lon']].drop_duplicates('mkt_name')
        print(f"  Unmatched: {unmatch_mkts['mkt_name'].tolist()}")
    
    price_joined_parts.append(joined)

price_joined = pd.concat(price_joined_parts, ignore_index=True)

# Aggregate: mean price per admin2/year/month
price_agg = price_joined.dropna(subset=['admin2_canonical']).groupby(
    ['year', 'month', 'admin2_canonical']
)[price_cols].mean().reset_index()

print(f"\nPrice aggregated: {len(price_agg)} rows")
print(f"Admin2 with price: {price_agg['admin2_canonical'].nunique()}")

## 5. Process Population Data (WorldPop Zonal Stats)

In [ ]:
pop_dfs = []
raster_map = {
    'KEN': 'ken_pop_2020_1km.tif',
    'SOM': 'som_pop_2020_1km.tif',
}

for iso in iso3_list:
    raster_path = os.path.join(DATA_DIR, 'population_worldpop', raster_map[iso])
    if os.path.exists(raster_path):
        iso_pop = process_worldpop_population(admin_gdf, raster_path, iso)
        if not iso_pop.empty:
            pop_dfs.append(iso_pop)
    else:
        logger.warning(f"Population raster not found: {raster_path}")

pop_agg = pd.concat(pop_dfs, ignore_index=True)
pop_agg = pop_agg.groupby('shapeName')['population'].sum().reset_index()
pop_agg.rename(columns={'shapeName': 'admin2_canonical'}, inplace=True)
print(f"Population: {len(pop_agg)} admin2 regions")

## 6. Process Crop Data

In [ ]:
crop_proc = crop_df[crop_df['shapeISO_ADM0'].isin(iso3_list)].copy()
canonical_set = set(canonical_names)

crop_proc['admin2_canonical'] = crop_proc['shapeName_ADM2'].where(
    crop_proc['shapeName_ADM2'].isin(canonical_set)
)

# Fuzzy match fallback for unmatched
unmatched_mask = crop_proc['admin2_canonical'].isna()
if unmatched_mask.any():
    fallback = fuzzy_match_names(crop_proc.loc[unmatched_mask, 'shapeName_ADM2'], canonical_names)
    crop_proc.loc[unmatched_mask, 'admin2_canonical'] = crop_proc.loc[unmatched_mask, 'shapeName_ADM2'].map(fallback)

crop_agg = crop_proc.dropna(subset=['admin2_canonical']).groupby(
    'admin2_canonical'
)['value'].mean().reset_index().rename(columns={'value': 'crop_cover_fraction'})

print(f"Crop data: {len(crop_agg)} admin2 regions")
print(f"Note: values are in % scale (1-83), NOT 0-1 fraction")

## 7. Process ACLED Conflict Data

In [ ]:
acled_joined = spatial_join_points(acled_df, admin_gdf, 'CENTROID_LONGITUDE', 'CENTROID_LATITUDE')

acled_joined['year'] = pd.to_datetime(acled_joined['WEEK']).dt.year
acled_joined['month'] = pd.to_datetime(acled_joined['WEEK']).dt.month

acled_agg = acled_joined.dropna(subset=['admin2_canonical']).groupby(
    ['year', 'month', 'admin2_canonical']
).agg({
    'FATALITIES': 'sum',
    'EVENTS': 'count'
}).reset_index().rename(columns={'EVENTS': 'conflict_events', 'FATALITIES': 'conflict_fatalities'})

print(f"ACLED aggregated: {len(acled_agg)} rows")
unmatched = acled_joined['admin2_canonical'].isna().sum()
print(f"Unmatched events: {unmatched} (mostly maritime)")

## 8. Final Merge

In [ ]:
# Create skeleton
years = sorted(price_df['year'].unique())
months = sorted(price_df['month'].unique())
master = create_master_skeleton(years, months, admin_gdf)
print(f"Skeleton: {master.shape} ({master['admin2'].nunique()} admin2 x {len(years)*len(months)} months)")

# Merge all
merged = pd.merge(master, price_agg,
    left_on=['year', 'month', 'admin2'], right_on=['year', 'month', 'admin2_canonical'], how='left')
merged.drop(columns=['admin2_canonical'], inplace=True, errors='ignore')

merged = pd.merge(merged, pop_agg, left_on='admin2', right_on='admin2_canonical', how='left')
merged.drop(columns=['admin2_canonical'], inplace=True, errors='ignore')

merged = pd.merge(merged, crop_agg, left_on='admin2', right_on='admin2_canonical', how='left')
merged.drop(columns=['admin2_canonical'], inplace=True, errors='ignore')

merged = pd.merge(merged, acled_agg,
    left_on=['year', 'month', 'admin2'], right_on=['year', 'month', 'admin2_canonical'], how='left')
merged.drop(columns=['admin2_canonical'], inplace=True, errors='ignore')

merged['conflict_events'] = merged['conflict_events'].fillna(0)
merged['conflict_fatalities'] = merged['conflict_fatalities'].fillna(0)

print(f"Final merged: {merged.shape}")
print(f"Columns: {list(merged.columns)}")
merged.head()

## 9. Save

In [ ]:
output_path = os.path.join(DATA_DIR, 'processed/subnational_merged_v2_KEN_SOM.parquet')
merged.to_parquet(output_path, index=False)
print(f"Saved to {output_path}")
print(f"Shape: {merged.shape}")

---
# Quality Checks

Below are systematic checks on the merged data.


## QC 1: Null / Missing Values

In [ ]:
null_counts = merged.isnull().sum()
null_pct = (merged.isnull().mean() * 100).round(2)
qc_null = pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})
print(qc_null)
print(f"\nTotal rows: {len(merged)}")

## QC 2: Admin2 Boundary Completeness

In [ ]:
for iso in iso3_list:
    gdf = gpd.read_file(os.path.join(boundary_dir, f'gb_{iso}_ADM2.geojson'))
    expected = set(gdf['shapeName'].unique())
    actual = set(merged[merged['country_iso'] == iso]['admin2'].unique())
    missing = expected - actual
    extra = actual - expected
    print(f"{iso}: expected={len(expected)}, actual={len(actual)}, "
          f"missing={len(missing)}, extra={len(extra)}")
    if missing:
        print(f"  Missing: {sorted(missing)}")

## QC 3: Price Coverage per Country

In [ ]:
for iso in iso3_list:
    sub = merged[merged['country_iso'] == iso]
    total_admin2 = sub['admin2'].nunique()
    for col in ['c_maize_fao', 'c_food_price_index', 'c_sorghum']:
        notnull = sub[col].notna().sum()
        admin2_with = sub[sub[col].notna()]['admin2'].nunique()
        print(f"{iso} {col}: {notnull}/{len(sub)} ({notnull/len(sub)*100:.1f}%), "
              f"{admin2_with}/{total_admin2} admin2")
    print()

## QC 4: Price Range & Outliers

In [ ]:
for col in ['c_maize_fao', 'c_food_price_index', 'c_sorghum']:
    valid = merged[col].dropna()
    q1, q3 = valid.quantile(0.25), valid.quantile(0.75)
    iqr = q3 - q1
    n_outliers = ((valid < q1 - 3*iqr) | (valid > q3 + 3*iqr)).sum()
    print(f"{col}:")
    print(f"  Range: {valid.min():.2f} ~ {valid.max():.2f}")
    print(f"  Mean: {valid.mean():.2f}, Median: {valid.median():.2f}")
    print(f"  Negative: {(valid < 0).sum()}, Zero: {(valid == 0).sum()}")
    print(f"  Extreme outliers (3*IQR): {n_outliers}")
    print()

print("NOTE: c_maize_fao unit differs by country (KES vs SOS).")
print("Consider using c_food_price_index (normalized 0-2 scale) for cross-country modeling.")

## QC 5: Cross-boundary Contamination Check

In [ ]:
# Verify no ETH data leaked in
print(f"ETH rows: {(merged['country_iso'] == 'ETH').sum()}")
print(f"Countries present: {merged['country_iso'].unique()}")
print(f"Duplicate (admin2, iso, year, month): {merged.duplicated(subset=['admin2','country_iso','year','month']).sum()}")

# Check admin2 names don't overlap between countries
cross = merged.groupby('admin2')['country_iso'].nunique()
shared = cross[cross > 1]
if len(shared):
    print(f"\nAdmin2 shared across countries: {list(shared.index)}")
else:
    print("\nNo admin2 names shared across countries.")

## QC 6: Skeleton Completeness

In [ ]:
yr_month_sizes = merged.groupby(['year', 'month']).size()
print(f"Year range: {merged['year'].min()} - {merged['year'].max()}")
print(f"Rows per year-month (should be constant): {yr_month_sizes.unique()}")

all_years = range(merged['year'].min(), merged['year'].max() + 1)
all_months = range(1, 13)
expected = set((y, m) for y in all_years for m in all_months)
actual = set(zip(merged['year'], merged['month']))
missing = expected - actual
if missing:
    print(f"Missing year-month combos: {sorted(missing)}")
else:
    print("All year-month combos present.")

## QC 7: Population / Crop / Conflict

In [ ]:
print("=== Population ===")
print(f"  Null: {merged['population'].isnull().sum()}")
print(f"  Range: {merged['population'].min():.0f} ~ {merged['population'].max():.0f}")
print(f"  Time-invariant: {merged.groupby(['admin2','country_iso'])['population'].nunique().eq(1).all()}")

print("\n=== Crop Cover Fraction ===")
print(f"  Null: {merged['crop_cover_fraction'].isnull().sum()} ({merged['crop_cover_fraction'].isnull().mean()*100:.1f}%)")
print(f"  Range: {merged['crop_cover_fraction'].dropna().min():.2f} ~ {merged['crop_cover_fraction'].dropna().max():.2f}")
print(f"  NOTE: Values are in % (1-83), not 0-1 fraction")

# Crop null admin2
crop_null = merged[merged['crop_cover_fraction'].isnull()].groupby('country_iso')['admin2'].unique()
print(f"\n  Null admin2:")
for iso, admins in crop_null.items():
    print(f"    {iso} ({len(admins)}): {list(admins)}")

print("\n=== Conflict ===")
print(f"  Rows with conflict > 0: {(merged['conflict_events'] > 0).sum()} ({(merged['conflict_events'] > 0).mean()*100:.1f}%)")
for iso in iso3_list:
    sub = merged[merged['country_iso'] == iso]
    print(f"  {iso}: {(sub['conflict_events'] > 0).mean()*100:.1f}% rows have conflict")

## QC 8: KEN vs SOM Price Unit Comparison

In [ ]:
print("c_maize_fao is in LOCAL CURRENCY (KES for Kenya, SOS for Somalia)")
print("c_food_price_index is NORMALIZED (around 1.0)\n")

for col in ['c_maize_fao', 'c_food_price_index']:
    print(f"{col}:")
    for iso in iso3_list:
        v = merged[(merged['country_iso'] == iso) & (merged[col].notna())][col]
        if len(v) > 0:
            print(f"  {iso}: mean={v.mean():.2f}, median={v.median():.2f}, "
                  f"min={v.min():.2f}, max={v.max():.2f}")
    print()

## QC 9: Visual Checks (run these to inspect)

Run the cells below to generate maps and charts for visual inspection.


In [ ]:
# Price coverage map
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for idx, iso in enumerate(iso3_list):
    ax = axes[idx]
    gdf = gpd.read_file(os.path.join(boundary_dir, f'gb_{iso}_ADM2.geojson'))
    price_admins = set(merged[(merged['country_iso']==iso) & (merged['c_maize_fao'].notna())]['admin2'].unique())
    gdf['has_price'] = gdf['shapeName'].isin(price_admins)
    gdf.plot(column='has_price', ax=ax, legend=True, 
             cmap='RdYlGn', edgecolor='gray', linewidth=0.3,
             legend_kwds={'labels': ['No price data', 'Has price data']})
    ax.set_title(f'{iso}: Price Data Coverage ({len(price_admins)} admin2)')
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# c_food_price_index time series (sample admin2)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for idx, iso in enumerate(iso3_list):
    ax = axes[idx]
    sub = merged[(merged['country_iso']==iso) & (merged['c_food_price_index'].notna())]
    sample_admins = sub['admin2'].unique()[:5]
    for admin2 in sample_admins:
        ts = sub[sub['admin2']==admin2].sort_values(['year','month'])
        ts['date'] = pd.to_datetime(ts[['year','month']].assign(day=1))
        ax.plot(ts['date'], ts['c_food_price_index'], label=admin2, alpha=0.7)
    ax.set_title(f'{iso}: Food Price Index (sample admin2)')
    ax.set_ylabel('c_food_price_index')
    ax.legend(fontsize=7, loc='upper left')

plt.tight_layout()
plt.show()